# 数据库约束设计实习
本次实习的目标是体验如何在数据库中利用各种手段完成数据库约束设计。
## 实习任务
我们需要为以下两个表设计约束
- Emp(<u>eno</u>, ename, birtyday, level, position, salary, dno)
- Dept(<u>dno</u>, dname, budget, manager)

约束要求：
1. eno和dno是递增序列号形式的主键，长度为4的整型，格式为0001、0002等
2. Emp中的dno为参照Dept的外键，Dept的manager为参照Emp的外键
3. 测试外键定义的三种形式
4. 限定dname为枚举类型（数学学院、计算机学院、智能学院、电子学院、元培学院）
5. 限定position为枚举类型（教师、教务、会计、秘书）\n
6. 限定level为1到5，默认值为3，salary为2000到200000"

In [1]:
pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:

# 数据库约束设计实习
import os
import sqlalchemy
from sqlalchemy import create_engine, text
from sqlalchemy import MetaData, Table, Column, Integer, String, Float, Date, ForeignKey, CheckConstraint
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker, relationship
from sqlalchemy.sql import func
import pandas as pd
from datetime import date

In [3]:
# 创建SQLite数据库连接
engine = create_engine('sqlite:///employee_dept.db', echo=True)
metadata = MetaData()
Base = declarative_base()

# 创建会话
Session = sessionmaker(bind=engine)
session = Session()

C:\Users\Dai\AppData\Local\Temp\ipykernel_30056\1065261567.py:4: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


In [4]:
# 如果已经存在数据库，删除它
if os.path.exists('employee_dept.db'):
    os.remove('employee_dept.db')
    print("删除现有数据库文件")

删除现有数据库文件


In [11]:
# 定义Emp表（放在前面，因为Dept表需要引用它）
class Emp(Base):
    __tablename__ = 'emp'  # 修正了双下划线
    
    eno = Column(String(4), primary_key=True)
    ename = Column(String(50))
    birtyday = Column(Date)
    level = Column(Integer, CheckConstraint("level BETWEEN 1 AND 5"), default=3)
    position = Column(String(10), CheckConstraint("position IN ('教师', '教务', '会计', '秘书')"))
    salary = Column(Float, CheckConstraint("salary BETWEEN 2000 AND 200000"))
    dno = Column(String(4), ForeignKey('dept.dno', deferrable=True, initially='DEFERRED'))

    def __repr__(self):
        return f"<Emp(eno='{self.eno}', ename='{self.ename}', level={self.level}, position='{self.position}', salary={self.salary}, dno='{self.dno}')>"
    
    # 注意：我们将在Dept类定义后添加关系

# 定义Dept表
class Dept(Base):
    __tablename__ = 'dept'  # 修正了双下划线
    
    dno = Column(String(4), primary_key=True)
    dname = Column(String(20), CheckConstraint("dname IN ('数学学院', '计算机学院', '智能学院', '电子学院', '元培学院')"))
    budget = Column(Float)
    manager = Column(String(4), ForeignKey('emp.eno', deferrable=True, initially='DEFERRED'))

    def __repr__(self):
        return f"<Dept(dno='{self.dno}', dname='{self.dname}', budger={self.budget}, manager='{self.manager}')>"

# 添加关系引用，解决循环引用问题
Emp.department = relationship("Dept", foreign_keys=[Emp.dno], backref="employees")
Dept.manager_emp = relationship("Emp", foreign_keys=[Dept.manager])


C:\Users\Dai\AppData\Local\Temp\ipykernel_30056\2343246097.py:2: SAWarning: This declarative base already contains a class with the same class name and module name as __main__.Emp, and will be replaced in the string-lookup table.
  class Emp(Base):


InvalidRequestError: Table 'emp' is already defined for this MetaData instance.  Specify 'extend_existing=True' to redefine options and columns on an existing Table object.

In [6]:
# 创建表
Base.metadata.create_all(engine)

2025-04-17 16:29:43,098 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-17 16:29:43,099 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("emp")
2025-04-17 16:29:43,100 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-04-17 16:29:43,102 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("emp")
2025-04-17 16:29:43,103 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-04-17 16:29:43,104 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("dept")
2025-04-17 16:29:43,105 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-04-17 16:29:43,107 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("dept")
2025-04-17 16:29:43,108 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-04-17 16:29:43,112 INFO sqlalchemy.engine.Engine 
CREATE TABLE emp (
	eno VARCHAR(4) NOT NULL, 
	ename VARCHAR(50), 
	birtyday DATE, 
	level INTEGER CHECK (level BETWEEN 1 AND 5), 
	position VARCHAR(10) CHECK (position IN ('教师', '教务', '会计', '秘书')), 
	salary FLOAT CHECK (salary BETWEEN 2000 AND 200000), 
	dno VARCHAR(

In [7]:
# 为SQLite创建序列模拟功能
def get_next_id(table_name):
    max_id_query = text(f"SELECT MAX(CAST(SUBSTR({table_name[0]}no, 1, 4) AS INTEGER)) FROM {table_name}")
    result = engine.execute(max_id_query).scalar()
    next_id = 1 if result is None else result + 1
    return f"{next_id:04d}"

In [8]:
# 插入初始数据
try:
    session.begin()
    dept1 = Dept(dno="0001", dname="计算机学院", budget=1000000)
    dept2 = Dept(dno="0002", dname="数学学院", budget=800000)
    dept3 = Dept(dno="0003", dname="智能学院", budget=1200000)
    session.add_all([dept1, dept2, dept3])
    session.flush()

    emp1 = Emp(eno="0001", ename="张三", birtyday=date(1980, 1, 15), level=4, position="教师", salary=15000, dno="0001")
    emp2 = Emp(eno="0002", ename="李四", birtyday=date(1985, 3, 20), level=3, position="教务", salary=8000, dno="0002")
    emp3 = Emp(eno="0003", ename="王五", birtyday=date(1990, 7, 10), level=5, position="教师", salary=20000, dno="0003")
    session.add_all([emp1, emp2, emp3])
    session.flush()

    dept1.manager = "0001"
    dept2.manager = "0002"
    dept3.manager = "0003"

    session.commit()
except Exception as e:
    session.rollback()
    print(f"插入数据时出错: {e}")

2025-04-17 16:29:47,235 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-17 16:29:47,239 INFO sqlalchemy.engine.Engine INSERT INTO dept (dno, dname, budget, manager) VALUES (?, ?, ?, ?)
2025-04-17 16:29:47,239 INFO sqlalchemy.engine.Engine [generated in 0.00095s] [('0001', '计算机学院', 1000000.0, None), ('0002', '数学学院', 800000.0, None), ('0003', '智能学院', 1200000.0, None)]
2025-04-17 16:29:47,253 INFO sqlalchemy.engine.Engine INSERT INTO emp (eno, ename, birtyday, level, position, salary, dno) VALUES (?, ?, ?, ?, ?, ?, ?)
2025-04-17 16:29:47,255 INFO sqlalchemy.engine.Engine [generated in 0.00171s] [('0001', '张三', '1980-01-15', 4, '教师', 15000.0, '0001'), ('0002', '李四', '1985-03-20', 3, '教务', 8000.0, '0002'), ('0003', '王五', '1990-07-10', 5, '教师', 20000.0, '0003')]
2025-04-17 16:29:47,259 INFO sqlalchemy.engine.Engine UPDATE dept SET manager=? WHERE dept.dno = ?
2025-04-17 16:29:47,260 INFO sqlalchemy.engine.Engine [generated in 0.00152s] [('0001', '0001'), ('0002', '0002'), ('0003', '00

In [9]:
# 测试默认值
try:
    session.begin()
    emp_default = Emp(eno="0004", ename="吴十", birtyday=date(1991, 8, 25), position="秘书", salary=7000, dno="0003")
    session.add(emp_default)
    session.commit()
    emp_result = session.query(Emp).filter_by(ename="吴十").first()
    print(f"默认值测试结果: {emp_result.level}")
except Exception as e:
    session.rollback()
    print(f"测试默认值时出错: {e}")

2025-04-17 16:29:52,694 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-17 16:29:52,694 INFO sqlalchemy.engine.Engine INSERT INTO emp (eno, ename, birtyday, level, position, salary, dno) VALUES (?, ?, ?, ?, ?, ?, ?)
2025-04-17 16:29:52,694 INFO sqlalchemy.engine.Engine [generated in 0.00114s] ('0004', '吴十', '1991-08-25', 3, '秘书', 7000.0, '0003')
2025-04-17 16:29:52,706 INFO sqlalchemy.engine.Engine COMMIT
2025-04-17 16:29:52,717 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-17 16:29:52,720 INFO sqlalchemy.engine.Engine SELECT emp.eno AS emp_eno, emp.ename AS emp_ename, emp.birtyday AS emp_birtyday, emp.level AS emp_level, emp.position AS emp_position, emp.salary AS emp_salary, emp.dno AS emp_dno 
FROM emp 
WHERE emp.ename = ?
 LIMIT ? OFFSET ?
2025-04-17 16:29:52,720 INFO sqlalchemy.engine.Engine [generated in 0.00111s] ('吴十', 1, 0)
默认值测试结果: 3


In [10]:
# 查询部门数据
for dept in session.query(Dept).all():
    print(dept)

2025-04-17 16:29:54,757 INFO sqlalchemy.engine.Engine SELECT dept.dno AS dept_dno, dept.dname AS dept_dname, dept.budget AS dept_budget, dept.manager AS dept_manager 
FROM dept
2025-04-17 16:29:54,758 INFO sqlalchemy.engine.Engine [generated in 0.00124s] ()


AttributeError: 'Dept' object has no attribute 'eno'

In [15]:
# 查询员工数据
for emp in session.query(Emp).all():
    print(emp)

# 清理资源
session.close()
print("会话已关闭")

2025-04-17 16:22:49,334 INFO sqlalchemy.engine.Engine SELECT emp.eno AS emp_eno, emp.ename AS emp_ename, emp.birtyday AS emp_birtyday, emp.level AS emp_level, emp.position AS emp_position, emp.salary AS emp_salary, emp.dno AS emp_dno 
FROM emp
2025-04-17 16:22:49,336 INFO sqlalchemy.engine.Engine [generated in 0.00206s] ()
2025-04-17 16:22:49,341 INFO sqlalchemy.engine.Engine ROLLBACK
会话已关闭
